<a href="https://colab.research.google.com/github/6eyeuser/6eyeuser/blob/main/ai-reseach-paper-summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies
!pip install flask flask-cors pdfplumber transformers torch pyngrok

# Mount your notebook or just run the service directly
from pyngrok import ngrok
import threading

# Your app.py code goes here (or upload the file)
# Then start Flask in a thread and expose it:
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

# Run Flask
!python app.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.9 MB/s eta 0:00:00


ERROR:pyngrok.process.ngrok:t=2026-03-12T21:54:32+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-12T21:54:32+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [1]:
# Step 1 — Install only what's needed (ngrok NOT required)
!pip install flask flask-cors pdfplumber transformers torch pyngrok -q

# Step 2 — Write app.py to disk
%%writefile app.py
# ... (your app.py content here)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.2 MB/s eta 0:00:00


UsageError: Line magic function `%%writefile` not found.


In [2]:
!pip install flask flask-cors pdfplumber transformers torch -q


In [4]:
%%writefile app.py
import io, logging
import pdfplumber
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import pipeline

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

log.info("Loading model...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0)
log.info("Model ready.")

app = Flask(__name__)
CORS(app)

def extract_text(pdf_bytes):
    parts = []
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                parts.append(t)
    text = "\n\n".join(parts).strip()
    if not text:
        raise ValueError("No text found — PDF may be a scanned image.")
    return text

def chunk_text(text, size=3000):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for p in paras:
        if len(current) + len(p) + 2 <= size:
            current += ("\n\n" if current else "") + p
        else:
            if current: chunks.append(current)
            current = p
    if current: chunks.append(current)
    return chunks

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

@app.route("/summarize", methods=["POST"])
def summarize():
    if "pdf" not in request.files:
        return jsonify({"error": "No pdf field"}), 400
    pdf_bytes = request.files["pdf"].read()
    try:
        text = extract_text(pdf_bytes)
    except ValueError as e:
        return jsonify({"error": str(e)}), 422
    chunks = chunk_text(text)[:5]
    summaries = []
    for i, chunk in enumerate(chunks):
        log.info(f"Summarizing chunk {i+1}/{len(chunks)}")
        result = summarizer(chunk, max_length=180, min_length=40,
                            do_sample=False, truncation=True)
        summaries.append(result[0]["summary_text"])
    final = " ".join(summaries)
    if len(summaries) > 1:
        final = summarizer(final, max_length=250, min_length=60,
                           do_sample=False, truncation=True)[0]["summary_text"]
    return jsonify({"summary": final})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000, debug=False)

Overwriting app.py


In [5]:
import subprocess, threading, time

# Start Flask in background
def run():
    subprocess.run(["python", "app.py"])

threading.Thread(target=run, daemon=True).start()

# Wait for model to load (takes ~30 seconds first time)
print("Loading model, please wait...")
time.sleep(35)

# Get Colab's built-in public URL — no account needed
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8000)")
print("\n✅ AI Service URL:", url)
print("\nPaste this into backend/server.js as AI_SERVICE")

Loading model, please wait...

✅ AI Service URL: https://8000-gpu-t4-s-kkb-ass1c1-1h6vauhb8xwr-c.asia-southeast1-1.prod.colab.dev

Paste this into backend/server.js as AI_SERVICE
